# Câu 23
Viết thuật toán cho quy trình thuận/ngược của phương pháp Gauss giải hệ phương trình AX = B với A,B là các ma trận với cỡ lần lượt là m.n và m.p

## Quy trình thuận — Khử Gauss

### Bài toán
Biến đổi ma trận mở rộng về dạng bậc thang:

$$[A \mid B] \xrightarrow{\text{khử Gauss}} [U \mid B']$$

### Thuật toán

**Input:**
- $A$ — ma trận $m \times n$
- $B$ — ma trận $m \times p$

**Output:** $[U \mid B']$ — ma trận mở rộng dạng bậc thang

```
BEGIN
  C        ← [A | B]          // ma trận mở rộng m×(n+p)
  pivot_row ← 1

  FOR k = 1 TO n DO
    p ← argmax |C[i][k]|  với  i = pivot_row..m

    IF |C[p][k]| < tol THEN
      CONTINUE                 // cột k toàn 0, bỏ qua
    END IF

    IF p ≠ pivot_row THEN
      SWAP(C[pivot_row], C[p])
    END IF

    FOR i = pivot_row+1 TO m DO
      C[i] ← C[i] − (C[i][k] / C[pivot_row][k]) · C[pivot_row]
    END FOR

    pivot_row ← pivot_row + 1
  END FOR

  PRINT [U | B']  với  U ← C[:, 1..n],  B' ← C[:, n+1..n+p]
END
```

### Lưu ý
- Vòng lặp duyệt theo **cột của $A$**, mỗi phép khử cập nhật toàn bộ hàng của $[A \mid B]$
- Kết thúc: $U$ bậc thang, $r =$ số pivot tìm được = $\text{rank}(A)$

# Câu 24
Viết thuật toán cho quy trình ngược của phương pháp Gauss giải hệ phương trình AX = B với kích cỡ lần lượt là m.n và n.p

## Quy trình ngược — Thế ngược

### Bài toán
Từ $[U \mid B']$ sau quy trình thuận, tìm $X$ cỡ $n \times p$ thỏa $UX = B'$.

### Các trường hợp nghiệm

Gọi $r = \text{rank}(A)$:

| Điều kiện | Kết luận |
|:---|:---|
| $\exists$ hàng $[\,0 \cdots 0 \mid b_i' \neq 0\,]$ | Hệ **vô nghiệm** |
| $r = n$ | Nghiệm **duy nhất** |
| $r < n$ | Nghiệm **vô số** ($n - r$ ẩn tự do) |

### Thuật toán

**Input:**
- $U$ — ma trận $m \times n$ bậc thang, $r$ — hạng
- $B'$ — ma trận $m \times p$

**Output:** $X$ — ma trận nghiệm $n \times p$

```
BEGIN
  // Kiểm tra vô nghiệm
  FOR i = r+1 TO m DO
    IF ||B'[i]|| > tol THEN  STOP "Hệ vô nghiệm"
  END FOR

  // Xác định ẩn pivot và ẩn tự do
  pivot_cols ← cột chứa pivot        // r cột
  free_cols  ← cột không có pivot    // n−r cột

  // Gán ẩn tự do
  FOR j ∈ free_cols DO
    X[j] ← 0     // hoặc tham số t₁, t₂, ...
  END FOR

  // Thế ngược
  FOR i = r DOWNTO 1 DO
    k ← pivot_cols[i]
    FOR s = 1 TO p DO
      X[k][s] ← ( B'[i][s] − Σⱼ U[i][j]·X[j][s], j=k+1..n ) / U[i][k]
    END FOR
  END FOR

  PRINT X
END
```

# Câu 25
Áp dụng 2 thuật toán giải hệ phương trình với dữ liệu ma trận tự giả lập (dùng code Python)

In [8]:
import numpy as np

def gauss_solve(A, B):
    """Giải hệ AX = B bằng khử Gauss có chọn pivot. B có thể là vector hoặc ma trận."""
    A = A.astype(float)
    B = np.array(B, dtype=float)
    if B.ndim == 1:
        B = B.reshape(-1, 1)
    m, n = A.shape
    p = B.shape[1]
    C = np.hstack([A, B])

    def print_matrix(C, label):
        sep = "-" * (13 * (n + p) + 3)
        print(f"\n{label}:")
        print(sep)
        for row in C:
            vals = "  ".join(f"{v:10.4f}" for v in row[:n])
            rhs  = "  ".join(f"{v:10.4f}" for v in row[n:])
            print(f"  {vals}  |  {rhs}")
        print(sep)

    print_matrix(C, "Ma trận ban đầu [A | B]")

    # ── Quy trình thuận ──────────────────────────────────────────
    pivot_row, pivot_cols = 0, []
    step = 0
    for k in range(n):
        i_max = pivot_row + np.argmax(np.abs(C[pivot_row:, k]))
        if abs(C[i_max, k]) < 1e-12:
            continue
        if i_max != pivot_row:
            C[[pivot_row, i_max]] = C[[i_max, pivot_row]]
            print(f"  → Hoán đổi hàng {pivot_row+1} ↔ hàng {i_max+1}")
        for i in range(pivot_row + 1, m):
            f = C[i, k] / C[pivot_row, k]
            C[i] -= f * C[pivot_row]
            print(f"  → h{i+1} ← h{i+1} − ({f:.4f})·h{pivot_row+1}")
        pivot_cols.append(k)
        pivot_row += 1
        step += 1
        print_matrix(C, f"Bước {step} (khử cột {k+1})")

    U, Bp = C[:, :n], C[:, n:]
    r = len(pivot_cols)

    # ── Kiểm tra vô nghiệm ───────────────────────────────────────
    for i in range(r, m):
        if np.max(np.abs(Bp[i])) > 1e-12:
            print("\nHệ VÔ NGHIỆM")
            return None

    # ── Quy trình ngược ──────────────────────────────────────────
    X = np.zeros((n, p))
    print("\n=== Quy trình ngược ===")
    for i in range(r - 1, -1, -1):
        k = pivot_cols[i]
        X[k] = (Bp[i] - U[i, k+1:] @ X[k+1:]) / U[i, k]
        vals = "  ".join(f"{v:.6f}" for v in X[k])
        print(f"  x{k+1} = [{vals}]")

    free = [j for j in range(n) if j not in pivot_cols]
    if free:
        print(f"Ẩn tự do: x{[j+1 for j in free]} = 0 (mặc định)")

    return X


# ── Chạy thử: hệ 4×4, B có 2 cột ────────────────────────────────
A = np.array([[ 2,  1,  0,  3],
              [ 4,  3, -1,  5],
              [-2,  0,  2,  1],
              [ 0,  1, -1,  2]])

B = np.array([[1,  2],
              [3,  5],
              [0, -1],
              [2,  1]])

X = gauss_solve(A, B)
if X is not None:
    print("\n=== Nghiệm X ===")
    print(X.round(6))
    print("\nKiểm tra A·X (phải bằng B):")
    print((A @ X).round(6))


Ma trận ban đầu [A | B]:
---------------------------------------------------------------------------------
      2.0000      1.0000      0.0000      3.0000  |      1.0000      2.0000
      4.0000      3.0000     -1.0000      5.0000  |      3.0000      5.0000
     -2.0000      0.0000      2.0000      1.0000  |      0.0000     -1.0000
      0.0000      1.0000     -1.0000      2.0000  |      2.0000      1.0000
---------------------------------------------------------------------------------
  → Hoán đổi hàng 1 ↔ hàng 2
  → h2 ← h2 − (0.5000)·h1
  → h3 ← h3 − (-0.5000)·h1
  → h4 ← h4 − (0.0000)·h1

Bước 1 (khử cột 1):
---------------------------------------------------------------------------------
      4.0000      3.0000     -1.0000      5.0000  |      3.0000      5.0000
      0.0000     -0.5000      0.5000      0.5000  |     -0.5000     -0.5000
      0.0000      1.5000      1.5000      3.5000  |      1.5000      1.5000
      0.0000      1.0000     -1.0000      2.0000  |      2.0000    

# Phương pháp Gauss và Gauss-Jordan giải hệ phương trình tuyến tính

## Bài toán
Giải hệ $n$ phương trình tuyến tính $n$ ẩn:

$$A\mathbf{x} = \mathbf{b}$$

tức là:

$$\begin{pmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\ a_{21} & a_{22} & \cdots & a_{2n} \\ \vdots & \vdots & \ddots & \vdots \\ a_{n1} & a_{n2} & \cdots & a_{nn} \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{pmatrix} = \begin{pmatrix} b_1 \\ b_2 \\ \vdots \\ b_n \end{pmatrix}$$

---

## 1. Phương pháp Gauss cơ bản

### Ý tưởng
Biến đổi ma trận mở rộng $[A|\mathbf{b}]$ về dạng **tam giác trên**, rồi giải ngược.

### Hai pha

**Pha 1 — Khử xuôi:** Biến $A$ về tam giác trên

$$[A|\mathbf{b}] \rightarrow [U|\mathbf{b}']$$

$$\begin{pmatrix} a_{11} & a_{12} & \cdots & a_{1n} \\ 0 & a_{22}' & \cdots & a_{2n}' \\ \vdots & \ddots & \ddots & \vdots \\ 0 & \cdots & 0 & a_{nn}' \end{pmatrix} \begin{pmatrix} b_1' \\ b_2' \\ \vdots \\ b_n' \end{pmatrix}$$

Công thức khử tại bước $k$:

$$m_{ik} = \frac{a_{ik}^{(k)}}{a_{kk}^{(k)}} \quad \text{(hệ số nhân)}$$

$$a_{ij}^{(k+1)} = a_{ij}^{(k)} - m_{ik} \cdot a_{kj}^{(k)}, \quad j = k, \ldots, n$$

$$b_i^{(k+1)} = b_i^{(k)} - m_{ik} \cdot b_k^{(k)}, \quad i = k+1, \ldots, n$$

**Pha 2 — Thế ngược:** Giải từ dưới lên

$$x_n = \frac{b_n'}{a_{nn}'}$$

$$x_i = \frac{1}{a_{ii}'}\left(b_i' - \sum_{j=i+1}^{n} a_{ij}' x_j\right), \quad i = n-1, \ldots, 1$$

### Thuật toán

```
Input:  A (ma trận n×n), b (vector n)
Output: x (nghiệm)

BEGIN
  // Pha 1: Khử xuôi
  FOR k = 1 TO n-1 DO
    FOR i = k+1 TO n DO
      m    ← A[i][k] / A[k][k]         // hệ số nhân
      FOR j = k TO n DO
        A[i][j] ← A[i][j] - m * A[k][j]
      END FOR
      b[i] ← b[i] - m * b[k]
    END FOR
  END FOR

  // Pha 2: Thế ngược
  x[n] ← b[n] / A[n][n]
  FOR i = n-1 DOWNTO 1 DO
    s    ← b[i]
    FOR j = i+1 TO n DO
      s ← s - A[i][j] * x[j]
    END FOR
    x[i] ← s / A[i][i]
  END FOR

  Print x
END
```

> ⚠️ Nếu $a_{kk} = 0$ tại bước $k$ thì không thể chia — cần chọn phần tử chủ!

---

## 2. Phương pháp Gauss có chọn phần tử chủ (Partial Pivoting)

### Ý tưởng
Trước mỗi bước khử $k$, **hoán đổi hàng** để đưa phần tử lớn nhất trong cột $k$ lên vị trí $a_{kk}$.

$$\text{Chọn } p = \arg\max_{i \geq k} |a_{ik}|, \quad \text{hoán đổi hàng } k \leftrightarrow p$$

### Tại sao cần?
- Tránh chia cho $a_{kk} \approx 0$ → mất ổn định số
- Giảm sai số làm tròn
- Phần tử chủ càng lớn → hệ số nhân $|m_{ik}| \leq 1$ → không khuếch đại sai số

### Thuật toán

```
Input:  A (ma trận n×n), b (vector n)
Output: x (nghiệm)

BEGIN
  // Pha 1: Khử xuôi CÓ chọn phần tử chủ
  FOR k = 1 TO n-1 DO

    // Tìm phần tử chủ
    p ← k
    FOR i = k+1 TO n DO
      IF |A[i][k]| > |A[p][k]| THEN p ← i
    END FOR

    // Kiểm tra suy biến
    IF |A[p][k]| < tol THEN
      STOP "Ma trận suy biến hoặc gần suy biến"
    END IF

    // Hoán đổi hàng k và p
    IF p ≠ k THEN
      SWAP(A[k], A[p])
      SWAP(b[k], b[p])
    END IF

    // Khử như Gauss cơ bản
    FOR i = k+1 TO n DO
      m    ← A[i][k] / A[k][k]
      FOR j = k TO n DO
        A[i][j] ← A[i][j] - m * A[k][j]
      END FOR
      b[i] ← b[i] - m * b[k]
    END FOR

  END FOR

  // Pha 2: Thế ngược (giống Gauss cơ bản)
  x[n] ← b[n] / A[n][n]
  FOR i = n-1 DOWNTO 1 DO
    s ← b[i]
    FOR j = i+1 TO n DO
      s ← s - A[i][j] * x[j]
    END FOR
    x[i] ← s / A[i][i]
  END FOR

  Print x
END
```

---

## 3. Phương pháp Gauss-Jordan

### Ý tưởng
Thay vì chỉ khử xuôi rồi thế ngược, Gauss-Jordan **khử cả lên lẫn xuống** để đưa $A$ về dạng **ma trận đơn vị**:

$$[A|\mathbf{b}] \rightarrow [I|\mathbf{x}]$$

Kết quả đọc trực tiếp từ cột $\mathbf{b}$ mà **không cần thế ngược**.

### So sánh với Gauss

$$\text{Gauss: } [A|\mathbf{b}] \rightarrow [U|\mathbf{b}'] \rightarrow \text{thế ngược} \rightarrow \mathbf{x}$$

$$\text{Gauss-Jordan: } [A|\mathbf{b}] \rightarrow [I|\mathbf{x}]$$

### Thuật toán

```
Input:  A (ma trận n×n), b (vector n)
Output: x (nghiệm, đọc trực tiếp từ b sau khi khử)

BEGIN
  FOR k = 1 TO n DO

    // Chọn phần tử chủ (nên có)
    p ← argmax(|A[i][k]|) với i = k..n
    IF p ≠ k THEN SWAP(A[k], A[p]); SWAP(b[k], b[p])

    // Chuẩn hóa hàng k (chia cho phần tử chủ)
    pivot ← A[k][k]
    FOR j = k TO n DO
      A[k][j] ← A[k][j] / pivot
    END FOR
    b[k] ← b[k] / pivot

    // Khử CẢ lên lẫn xuống
    FOR i = 1 TO n DO
      IF i ≠ k THEN
        m ← A[i][k]
        FOR j = k TO n DO
          A[i][j] ← A[i][j] - m * A[k][j]
        END FOR
        b[i] ← b[i] - m * b[k]
      END IF
    END FOR

  END FOR

  // Nghiệm chính là vector b
  x ← b
  Print x
END
```

---

## So sánh ba phương pháp

| | Gauss cơ bản | Gauss + pivoting | Gauss-Jordan |
|---|---|---|---|
| Kết quả trung gian | Tam giác trên | Tam giác trên | Ma trận đơn vị |
| Cần thế ngược? | Có | Có | Không |
| Ổn định số | Kém | Tốt | Tốt (nếu có pivoting) |
| Số phép tính | $\sim \dfrac{n^3}{3}$ | $\sim \dfrac{n^3}{3}$ | $\sim \dfrac{n^3}{2}$ |
| Dùng khi | Lý thuyết | Thực tế | Tìm $A^{-1}$ |

> ⚠️ Gauss-Jordan tốn hơn ~50% so với Gauss — trong thực tế ít dùng để giải hệ,
>    nhưng rất tiện để **tính ma trận nghịch đảo** $A^{-1}$ bằng cách mở rộng $[A|I] \rightarrow [I|A^{-1}]$.

In [9]:
import numpy as np

def gauss_jordan(A, B):
    """Giải AX = B bằng Gauss-Jordan — không cần thế ngược."""
    A = A.astype(float)
    B = np.array(B, dtype=float)
    if B.ndim == 1:
        B = B.reshape(-1, 1)
    m, n = A.shape
    p = B.shape[1]
    C = np.hstack([A, B])

    def print_matrix(C, label):
        sep = "-" * (13 * (n + p) + 3)
        print(f"\n{label}:")
        print(sep)
        for row in C:
            vals = "  ".join(f"{v:10.4f}" for v in row[:n])
            rhs  = "  ".join(f"{v:10.4f}" for v in row[n:])
            print(f"  {vals}  |  {rhs}")
        print(sep)

    print_matrix(C, "Ma trận ban đầu [A | B]")

    pivot_row, pivot_cols = 0, []
    step = 0
    for k in range(n):
        i_max = pivot_row + np.argmax(np.abs(C[pivot_row:, k]))
        if abs(C[i_max, k]) < 1e-12:
            continue
        if i_max != pivot_row:
            C[[pivot_row, i_max]] = C[[i_max, pivot_row]]
            print(f"  → Hoán đổi hàng {pivot_row+1} ↔ hàng {i_max+1}")

        # Chuẩn hóa hàng pivot
        C[pivot_row] /= C[pivot_row, k]
        print(f"  → h{pivot_row+1} ← h{pivot_row+1} / {C[pivot_row,k]:.4f} (chuẩn hóa)")

        # Khử cả trên lẫn dưới (điểm khác Gauss thường)
        for i in range(m):
            if i != pivot_row and abs(C[i, k]) > 1e-12:
                f = C[i, k]
                C[i] -= f * C[pivot_row]
                print(f"  → h{i+1} ← h{i+1} − ({f:.4f})·h{pivot_row+1}")

        pivot_cols.append(k)
        pivot_row += 1
        step += 1
        print_matrix(C, f"Bước {step} (khử cột {k+1})")

    r = len(pivot_cols)

    # Kiểm tra vô nghiệm
    for i in range(r, m):
        if np.max(np.abs(C[i, n:])) > 1e-12:
            print("\nHệ VÔ NGHIỆM")
            return None

    # Đọc nghiệm trực tiếp — không cần thế ngược
    X = np.zeros((n, p))
    for idx, k in enumerate(pivot_cols):
        X[k] = C[idx, n:]

    free = [j for j in range(n) if j not in pivot_cols]
    if free:
        print(f"Ẩn tự do: x{[j+1 for j in free]} = 0 (mặc định)")

    return X


# ── Chạy thử: cùng dữ liệu với gauss_solve ──────────────────────
A = np.array([[ 2,  1,  0,  3],
              [ 4,  3, -1,  5],
              [-2,  0,  2,  1],
              [ 0,  1, -1,  2]])

B = np.array([[1,  2],
              [3,  5],
              [0, -1],
              [2,  1]])

X = gauss_jordan(A, B)
if X is not None:
    print("\n=== Nghiệm X ===")
    print(X.round(6))
    print("\nKiểm tra A·X (phải bằng B):")
    print((A @ X).round(6))


Ma trận ban đầu [A | B]:
---------------------------------------------------------------------------------
      2.0000      1.0000      0.0000      3.0000  |      1.0000      2.0000
      4.0000      3.0000     -1.0000      5.0000  |      3.0000      5.0000
     -2.0000      0.0000      2.0000      1.0000  |      0.0000     -1.0000
      0.0000      1.0000     -1.0000      2.0000  |      2.0000      1.0000
---------------------------------------------------------------------------------
  → Hoán đổi hàng 1 ↔ hàng 2
  → h1 ← h1 / 1.0000 (chuẩn hóa)
  → h2 ← h2 − (2.0000)·h1
  → h3 ← h3 − (-2.0000)·h1

Bước 1 (khử cột 1):
---------------------------------------------------------------------------------
      1.0000      0.7500     -0.2500      1.2500  |      0.7500      1.2500
      0.0000     -0.5000      0.5000      0.5000  |     -0.5000     -0.5000
      0.0000      1.5000      1.5000      3.5000  |      1.5000      1.5000
      0.0000      1.0000     -1.0000      2.0000  |      2.0